# Problem Statement

The UCI News Aggregator dataset is a set of over 420,000 news articles that were compiled in 2014.  

Our goal for this analysis is to determine underlying trends among the articles to uncover commonalities and other links within the dataset using K-nearest neighbors, a machine learning technique that's usually used in analyzing unlabeled data.  However, in this case, it will be used to aid in further exploring the UCI News Aggregator dataset to uncover trends that we may not notice otherwise.

# Dictionary

- ID: the numeric ID of the article
- TITLE: the headline of the article
- URL: the URL of the article
- PUBLISHER: the publisher of the article
- CATEGORY: the category of the news item; one of:
    - e: entertainment
    - b: business
    - t: science and technology
    - m: health
- STORY: alphanumeric ID of the news story that the article discusses
- HOSTNAME: hostname where the article was posted
- TIMESTAMP: approximate timestamp of the article's publication, given in Unix time (seconds since midnight on Jan 1, 1970)

# Exploration

In the data exploration phase, we will cleanse the data to ensure suitable usage for modelling.

## Library Imports

We will import the NumPy, Pandas, Scikit-Learn Matplotlib, and Seaborn libraries to analyze, model and visualize our data.

Meanwhile, pickle and re will allow us to save our variables and perform internal operations within the system in which we perform this analysis.

In [ ]:
# !pip install seaborn
# !pip install --upgrade pip

In [1]:
# Import the packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re

from sklearn.feature_extraction.text import CountVectorizer

import pickle

import warnings
warnings.filterwarnings("ignore")

# Import the SentenceBERT model
from sentence_transformers import SentenceTransformer

# Classification model
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Post analysis
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE

# Import the 3D visualization libraries
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import plotly.io as pio

from sklearn.utils import resample

In [2]:
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/'
results_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/results/'

Mounted at /content/drive


In [3]:
# Import the datasets
convabuse_sa = pd.read_csv('/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/convabuse_sa_punc_imbalanced.csv')
dynamically_generated_hate_speech_sa = pd.read_csv('/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/dghs_sa_punc_imbalanced.csv')
mlma_hate_speech_english_sa = pd.read_csv('/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/mlma_hate_speech_sa_punc_imbalanced.csv')

# News Title Classifier

Now that we've explored the data in-depth, we can move forward to analyze the data and model a classifier to predict the classification of a news article based on its title.

This is, once again, a preview of the dataset after preprocessing.

In [4]:
convabuse_sa.head()

,annotator_id,prev_agent,prev_user,agent,user,bot,generalised,individual,system,explicit,implicit,abuse_level,target_groups,text,prev_agent_length,prev_user_length,agent_length,user_length,text_length,special_characters
0,7,Does that question interest you?,because my finger is happy,Why do you say your finger is happy?,do you rember japan,0,0,0,0,0,0,0,0,Does that question interest you? because my fi...,5,5,8,4,22,False
1,7,Does it please you to believe I am a nobe jocky?,bye bye,Please go on.,do you like it hard,0,0,0,0,0,0,1,0,Does it please you to believe I am a nobe jock...,11,2,3,5,21,False
2,7,Have you tried?,dick,Please go on.,oh what a pickle dear monastery,0,0,0,0,0,0,0,0,Have you tried? dick Please go on. oh what a p...,3,1,3,6,13,False
3,7,Will you be travelling in Economy Class?,yes,What city will you be flying from?,Sydney,1,0,0,0,0,0,0,0,Will you be travelling in Economy Class? yes W...,7,1,7,1,16,True
4,7,Please go on.,Hello.,How do you do. Please state your problem.,how old are you?,0,0,0,0,0,0,0,0,Please go on. Hello. How do you do. Please sta...,3,1,8,4,16,False


In [ ]:
dynamically_generated_hate_speech_sa.head()

,label,text,type,annotator,target_groups,original,text_length,special_characters
0,1.0,dalits are lowlives,0,3,0,1,3,False
1,1.0,gay people are a burden to our society,0,3,0,1,8,False
2,1.0,Arabs are not welcome,0,18,0,1,4,False
3,1.0,I\'m not saying we should actually eliminate h...,0,16,0,1,16,False
4,1.0,bananas are for black people,0,7,0,1,5,False


In [ ]:
mlma_hate_speech_english_sa.head()

,tweet,shock_annotator_sentiment,anger_annotator_sentiment,confusion_annotator_sentiment,sadness_annotator_sentiment,indifference_annotator_sentiment,disgust_annotator_sentiment,fear_annotator_sentiment,implicit,explicit,Abusive,Not Abusive,Very Abusive,Very Strongly Abusive,target_groups,tweet_length,special_characters
0,call sis im boy girl still faggot shut,0,0,1,0,0,1,0,1,0,1,0,0,0,2,8,False
1,@user @user legit nilas retarded idk,0,1,1,0,0,0,0,1,0,1,0,0,0,1,6,False
2,said retard @url,0,0,1,0,0,1,0,1,0,1,0,1,1,1,3,False
3,america another 8 years obama\'s ideology via ...,1,1,0,1,0,1,1,1,0,1,1,1,1,1,15,False
4,@user don... fucking made cry twat.,1,0,0,0,0,1,0,1,0,1,0,0,0,1,6,False


The following will reset the indices and eliminate the redundant article index column.

In [5]:
# Reset index
convabuse_sa.reset_index(drop=True, inplace=True)
# Drop prev_agent_
convabuse_sa.head()

,annotator_id,prev_agent,prev_user,agent,user,bot,generalised,individual,system,explicit,implicit,abuse_level,target_groups,text,prev_agent_length,prev_user_length,agent_length,user_length,text_length,special_characters
0,7,Does that question interest you?,because my finger is happy,Why do you say your finger is happy?,do you rember japan,0,0,0,0,0,0,0,0,Does that question interest you? because my fi...,5,5,8,4,22,False
1,7,Does it please you to believe I am a nobe jocky?,bye bye,Please go on.,do you like it hard,0,0,0,0,0,0,1,0,Does it please you to believe I am a nobe jock...,11,2,3,5,21,False
2,7,Have you tried?,dick,Please go on.,oh what a pickle dear monastery,0,0,0,0,0,0,0,0,Have you tried? dick Please go on. oh what a p...,3,1,3,6,13,False
3,7,Will you be travelling in Economy Class?,yes,What city will you be flying from?,Sydney,1,0,0,0,0,0,0,0,Will you be travelling in Economy Class? yes W...,7,1,7,1,16,True
4,7,Please go on.,Hello.,How do you do. Please state your problem.,how old are you?,0,0,0,0,0,0,0,0,Please go on. Hello. How do you do. Please sta...,3,1,8,4,16,False


In [ ]:
# Reset index
dynamically_generated_hate_speech_sa.reset_index(drop=True, inplace=True)
dynamically_generated_hate_speech_sa.head()

,label,text,type,annotator,target_groups,original,text_length,special_characters
0,1.0,dalits are lowlives,0,3,0,1,3,False
1,1.0,gay people are a burden to our society,0,3,0,1,8,False
2,1.0,Arabs are not welcome,0,18,0,1,4,False
3,1.0,I\'m not saying we should actually eliminate h...,0,16,0,1,16,False
4,1.0,bananas are for black people,0,7,0,1,5,False


In [ ]:
# Reset index
mlma_hate_speech_english_sa.reset_index(drop=True, inplace=True)
mlma_hate_speech_english_sa.head()

,tweet,shock_annotator_sentiment,anger_annotator_sentiment,confusion_annotator_sentiment,sadness_annotator_sentiment,indifference_annotator_sentiment,disgust_annotator_sentiment,fear_annotator_sentiment,implicit,explicit,Abusive,Not Abusive,Very Abusive,Very Strongly Abusive,target_groups,tweet_length,special_characters
0,call sis im boy girl still faggot shut,0,0,1,0,0,1,0,1,0,1,0,0,0,2,8,False
1,@user @user legit nilas retarded idk,0,1,1,0,0,0,0,1,0,1,0,0,0,1,6,False
2,said retard @url,0,0,1,0,0,1,0,1,0,1,0,1,1,1,3,False
3,america another 8 years obama\'s ideology via ...,1,1,0,1,0,1,1,1,0,1,1,1,1,1,15,False
4,@user don... fucking made cry twat.,1,0,0,0,0,1,0,1,0,1,0,0,0,1,6,False


## Univariate Modeling

Only use the title as the independent variable and the category as the dependent variable.

In [6]:
convabuse_sa.columns

Index(['annotator_id', 'prev_agent', 'prev_user', 'agent', 'user', 'bot',
       'generalised', 'individual', 'system', 'explicit', 'implicit',
       'abuse_level', 'target_groups', 'text', 'prev_agent_length',
       'prev_user_length', 'agent_length', 'user_length', 'text_length',
       'special_characters'],
      dtype='object')

In [7]:
# Create a new DataFrame with the text and category columns
knn_convabuse_df = pd.DataFrame({'prev_agent_text': convabuse_sa['prev_agent'], 'prev_user_text': convabuse_sa['prev_user'], 'agent_text': convabuse_sa['agent'], 'user_text': convabuse_sa['user'], 'abuse_level': convabuse_sa['abuse_level']})

# Merge the prev_agent_text, prev_user_text, agent_text, and user_text columns
knn_convabuse_df["text"] = knn_convabuse_df["prev_agent_text"].astype(str) + " " + knn_convabuse_df["prev_user_text"].astype(str) + " " + knn_convabuse_df["agent_text"].astype(str) + " " + knn_convabuse_df["user_text"].astype(str)

# Preview the DataFrame
knn_convabuse_df.head()

,prev_agent_text,prev_user_text,agent_text,user_text,abuse_level,text
0,Does that question interest you?,because my finger is happy,Why do you say your finger is happy?,do you rember japan,0,Does that question interest you? because my fi...
1,Does it please you to believe I am a nobe jocky?,bye bye,Please go on.,do you like it hard,1,Does it please you to believe I am a nobe jock...
2,Have you tried?,dick,Please go on.,oh what a pickle dear monastery,0,Have you tried? dick Please go on. oh what a p...
3,Will you be travelling in Economy Class?,yes,What city will you be flying from?,Sydney,0,Will you be travelling in Economy Class? yes W...
4,Please go on.,Hello.,How do you do. Please state your problem.,how old are you?,0,Please go on. Hello. How do you do. Please sta...


In [ ]:
# Create a new DataFrame with the text and category columns
knn_convabuse_prev_agent_df = pd.DataFrame({'prev_agent_text': convabuse_sa['prev_agent'], 'abuse_level': convabuse_sa['abuse_level']})

# Preview the DataFrame
knn_convabuse_prev_agent_df.head()

,prev_agent_text,abuse_level
0,Does that question interest you?,0
1,Does it please you to believe I am a nobe jocky?,1
2,Have you tried?,0
3,Will you be travelling in Economy Class?,0
4,Please go on.,0


In [ ]:
# Create a new DataFrame with the text and category columns
knn_convabuse_prev_user_df = pd.DataFrame({'prev_user_text': convabuse_sa['prev_user'], 'abuse_level': convabuse_sa['abuse_level']})

# Preview the DataFrame
knn_convabuse_prev_user_df.head()

,prev_user_text,abuse_level
0,because my finger is happy,0
1,bye bye,1
2,dick,0
3,yes,0
4,Hello.,0


In [ ]:
# Create a new DataFrame with the text and category columns
knn_convabuse_agent_df = pd.DataFrame({'agent_text': convabuse_sa['agent'], 'abuse_level': convabuse_sa['abuse_level']})

# Preview the DataFrame
knn_convabuse_agent_df.head()

,agent_text,abuse_level
0,Why do you say your finger is happy?,0
1,Please go on.,1
2,Please go on.,0
3,What city will you be flying from?,0
4,How do you do. Please state your problem.,0


In [ ]:
# Create a new DataFrame with the text and category columns
knn_convabuse_user_df = pd.DataFrame({'user_text': convabuse_sa['user'], 'abuse_level': convabuse_sa['abuse_level']})

# Preview the DataFrame
knn_convabuse_user_df.head()

,user_text,abuse_level
0,do you rember japan,0
1,do you like it hard,1
2,oh what a pickle dear monastery,0
3,Sydney,0
4,how old are you?,0


In [8]:
# Drop the previous columns
knn_convabuse_df = knn_convabuse_df.drop(columns=['prev_agent_text', 'prev_user_text', 'agent_text', 'user_text'])
knn_convabuse_df.head()

,abuse_level,text
0,0,Does that question interest you? because my fi...
1,1,Does it please you to believe I am a nobe jock...
2,0,Have you tried? dick Please go on. oh what a p...
3,0,Will you be travelling in Economy Class? yes W...
4,0,Please go on. Hello. How do you do. Please sta...


In [ ]:
# Create a new DataFrame with the text and category columns
knn_dghs_target_df = pd.DataFrame({'text': dynamically_generated_hate_speech_sa['text'], 'target_groups': dynamically_generated_hate_speech_sa['target_groups']})

# Preview the DataFrame
knn_dghs_target_df.head()

,text,target_groups
0,dalits are lowlives,0
1,gay people are a burden to our society,0
2,Arabs are not welcome,0
3,I\'m not saying we should actually eliminate h...,0
4,bananas are for black people,0


In [ ]:
# Create a new DataFrame with the text and category columns
knn_dghs_label_df = pd.DataFrame({'text': dynamically_generated_hate_speech_sa['text'], 'label': dynamically_generated_hate_speech_sa['label']})

# Preview the DataFrame
knn_dghs_label_df.head()

,text,label
0,dalits are lowlives,1.0
1,gay people are a burden to our society,1.0
2,Arabs are not welcome,1.0
3,I\'m not saying we should actually eliminate h...,1.0
4,bananas are for black people,1.0


In [ ]:
# Create a new DataFrame with the text and category columns
knn_mlma_hate_speech_target_df = pd.DataFrame({'tweet': mlma_hate_speech_english_sa['tweet'], 'target_groups': mlma_hate_speech_english_sa['target_groups']})

# Preview the DataFrame
knn_mlma_hate_speech_target_df.head()

,tweet,target_groups
0,call sis im boy girl still faggot shut,2
1,@user @user legit nilas retarded idk,1
2,said retard @url,1
3,america another 8 years obama\'s ideology via ...,1
4,@user don... fucking made cry twat.,1


Apply the train/test split in preparation for modeling.

In [9]:
# Train/test split
convabuse_X_train_titles, convabuse_X_test_titles, convabuse_y_train_titles, convabuse_y_test_titles = train_test_split(knn_convabuse_df[['text']], knn_convabuse_df['abuse_level'], test_size=0.2, random_state=42, stratify=knn_convabuse_df['abuse_level'])

# Check the distribution of the dependent variable in train vs test
convabuse_train_dist = knn_convabuse_df['text'].value_counts(normalize=True)
convabuse_test_dist = knn_convabuse_df['text'].value_counts(normalize=True)
print("Train Distribution:")
print(convabuse_train_dist)
print("\nTest Distribution:")
print(convabuse_test_dist)

Train Distribution:
text
_ _ _ hi                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     0.002624
Traveling, especially by airplane, usually emits greenhouse gases which are causing climate change. If you cannot avoid these emissions, you can buy \'offsets\', i.e. donations to projects that reduce greenhouse gases. A typical 3 hour flight will emit almost a ton of CO2 per economy passenger. If you cannot avoid flying, you can buy offsets for that carbon here:  [Buy Offsets](<URL>) I can also get you a more accurate estimate of your f

In [10]:
pd.Series(convabuse_y_train_titles).value_counts()

,count
abuse_level,
0,8050
3,694
2,599
1,526
4,192


In [ ]:
# Train/test split
convabuse_prev_agent_X_train_titles, convabuse_prev_agent_X_test_titles, convabuse_prev_agent_y_train_titles, convabuse_prev_agent_y_test_titles = train_test_split(knn_convabuse_prev_agent_df['prev_agent_text'], knn_convabuse_prev_agent_df['abuse_level'], test_size=0.2, random_state=42, stratify=knn_convabuse_prev_agent_df['abuse_level'])

# Check the distribution of the dependent variable in train vs test
convabuse_prev_agent_train_dist = knn_convabuse_prev_agent_df['prev_agent_text'].value_counts(normalize=True)
convabuse_prev_agent_test_dist = knn_convabuse_prev_agent_df['prev_agent_text'].value_counts(normalize=True)
print("Train Distribution:")
print(convabuse_prev_agent_train_dist)
print("\nTest Distribution:")
print(convabuse_prev_agent_test_dist)

Train Distribution:
prev_agent_text
_                                                              0.137235
Please go on.                                                  0.130715
You are sure?                                                  0.057804
Can you elaborate on that?                                     0.041425
Does that question interest you?                               0.038165
                                                                 ...   
Why do you say your bad?                                       0.000080
Would you prefer if I were not fucking retarded?               0.000080
Why do you not have anything to say?                           0.000080
Why do you say your childhood?                                 0.000080
Do you wish that u can read this u really need to get laid?    0.000080
Name: proportion, Length: 670, dtype: float64

Test Distribution:
prev_agent_text
_                                                              0.137235
Please go on.     

In [ ]:
pd.Series(convabuse_prev_agent_y_train_titles).value_counts()

,count
abuse_level,
0,8050
3,694
2,599
1,526
4,192


In [ ]:
# Train/test split
convabuse_prev_user_X_train_titles, convabuse_prev_user_X_test_titles, convabuse_prev_user_y_train_titles, convabuse_prev_user_y_test_titles = train_test_split(knn_convabuse_prev_user_df['prev_user_text'], knn_convabuse_prev_user_df['abuse_level'], test_size=0.2, random_state=42, stratify=knn_convabuse_prev_user_df['abuse_level'])

# Check the distribution of the dependent variable in train vs test
convabuse_prev_user_train_dist = knn_convabuse_prev_user_df['prev_user_text'].value_counts(normalize=True)
convabuse_prev_user_test_dist = knn_convabuse_prev_user_df['prev_user_text'].value_counts(normalize=True)
print("Train Distribution:")
print(convabuse_prev_user_train_dist)
print("\nTest Distribution:")
print(convabuse_prev_user_test_dist)

Train Distribution:
prev_user_text
_                                                              0.071400
yes                                                            0.056850
no                                                             0.026477
Yes                                                            0.019719
hi                                                             0.010575
                                                                 ...   
180cc                                                          0.000080
Pinar has such a sweet sexy ass                                0.000080
YEAH YOU SUCK DICK                                             0.000080
i am sorry,.                                                   0.000080
Because i believe you are one, yet you communicate with me.    0.000080
Name: proportion, Length: 2715, dtype: float64

Test Distribution:
prev_user_text
_                                                              0.071400
yes                

In [ ]:
pd.Series(convabuse_prev_user_y_train_titles).value_counts()

,count
abuse_level,
0,8050
3,694
2,599
1,526
4,192


In [ ]:
# Train/test split
convabuse_agent_X_train_titles, convabuse_agent_X_test_titles, convabuse_agent_y_train_titles, convabuse_agent_y_test_titles = train_test_split(knn_convabuse_agent_df['agent_text'], knn_convabuse_agent_df['abuse_level'], test_size=0.2, random_state=42, stratify=knn_convabuse_agent_df['abuse_level'])

# Check the distribution of the dependent variable in train vs test
convabuse_agent_train_dist = knn_convabuse_agent_df['agent_text'].value_counts(normalize=True)
convabuse_agent_test_dist = knn_convabuse_agent_df['agent_text'].value_counts(normalize=True)
print("Train Distribution:")
print(convabuse_agent_train_dist)
print("\nTest Distribution:")
print(convabuse_agent_test_dist)

Train Distribution:
agent_text
Please go on.                                                        0.148207
_                                                                    0.070764
You are sure?                                                        0.060269
Can you elaborate on that?                                           0.045003
You are being a bit negative.                                        0.044446
                                                                       ...   
Would you prefer if I were not a man or a woman?                     0.000080
Why do you tell me you were only four just now?                      0.000080
Why do you want to fuck her?                                         0.000080
How long have you been sorry?                                        0.000080
Why do you want it to be a german shepherd but their police dogs?    0.000080
Name: proportion, Length: 719, dtype: float64

Test Distribution:
agent_text
Please go on.                     

In [ ]:
pd.Series(convabuse_agent_y_train_titles).value_counts()

,count
abuse_level,
0,8050
3,694
2,599
1,526
4,192


In [ ]:
# Train/test split
convabuse_user_X_train_titles, convabuse_user_X_test_titles, convabuse_user_y_train_titles, convabuse_user_y_test_titles = train_test_split(knn_convabuse_user_df['user_text'], knn_convabuse_user_df['abuse_level'], test_size=0.2, random_state=42, stratify=knn_convabuse_user_df['abuse_level'])

# Check the distribution of the dependent variable in train vs test
convabuse_user_train_dist = knn_convabuse_user_df['user_text'].value_counts(normalize=True)
convabuse_user_test_dist = knn_convabuse_user_df['user_text'].value_counts(normalize=True)
print("Train Distribution:")
print(convabuse_user_train_dist)
print("\nTest Distribution:")
print(convabuse_user_test_dist)

Train Distribution:
user_text
yes                                               0.063211
no                                                0.026238
Yes                                               0.017731
christmas                                         0.014471
fuck you                                          0.010177
                                                    ...   
I really want to fuck her cute little shithole    0.000080
YES AND YOU ARE BLACK                             0.000080
You like girl on girl don\'t you?                 0.000080
im not to fond of blacks                          0.000080
Moron                                             0.000080
Name: proportion, Length: 2898, dtype: float64

Test Distribution:
user_text
yes                                               0.063211
no                                                0.026238
Yes                                               0.017731
christmas                                         0.014471
fuck you

In [ ]:
pd.Series(convabuse_user_y_train_titles).value_counts()

,count
abuse_level,
0,8050
3,694
2,599
1,526
4,192


In [ ]:
# Train/test split
dghs_target_X_train_titles, dghs_target_X_test_titles, dghs_target_y_train_titles, dghs_target_y_test_titles = train_test_split(knn_dghs_target_df['text'], knn_dghs_target_df['target_groups'], test_size=0.2, random_state=42, stratify=knn_dghs_target_df['target_groups'])

# Check the distribution of the dependent variable in train vs test
dghs_target_train_dist = knn_dghs_target_df['text'].value_counts(normalize=True)
dghs_target_test_dist = knn_dghs_target_df['text'].value_counts(normalize=True)
print("Train Distribution:")
print(dghs_target_train_dist)
print("\nTest Distribution:")
print(dghs_target_test_dist)

Train Distribution:
text
Blacks are such pondlife                                                                                                                                                                                                                                                                                                                                                                       0.000122
Muslims are a disgrace                                                                                                                                                                                                                                                                                                                                                                         0.000073
I endorse the genocide of Muslims                                                                                                                                                              

In [ ]:
pd.Series(dghs_target_y_train_titles).value_counts()

,count
target_groups,
0,24757
1,7992


In [ ]:
# Train/test split
dghs_label_X_train_titles, dghs_label_X_test_titles, dghs_label_y_train_titles, dghs_label_y_test_titles = train_test_split(knn_dghs_label_df['text'], knn_dghs_label_df['label'], test_size=0.2, random_state=42, stratify=knn_dghs_label_df['label'])

# Check the distribution of the dependent variable in train vs test
dghs_label_train_dist = knn_dghs_label_df['text'].value_counts(normalize=True)
dghs_label_test_dist = knn_dghs_label_df['text'].value_counts(normalize=True)
print("Train Distribution:")
print(dghs_label_train_dist)
print("\nTest Distribution:")
print(dghs_label_test_dist)

Train Distribution:
text
Blacks are such pondlife                                                                                                                                                                                                                                                                                                                                                                       0.000122
Muslims are a disgrace                                                                                                                                                                                                                                                                                                                                                                         0.000073
I endorse the genocide of Muslims                                                                                                                                                              

In [ ]:
pd.Series(dghs_label_y_train_titles).value_counts()

,count
label,
1.0,17574
0.0,15175


In [ ]:
# Train/test split
mlma_hate_speech_target_X_train_titles, mlma_hate_speech_target_X_test_titles, mlma_hate_speech_target_y_train_titles, mlma_hate_speech_target_y_test_titles = train_test_split(knn_mlma_hate_speech_target_df['tweet'], knn_mlma_hate_speech_target_df['target_groups'], test_size=0.2, random_state=42, stratify=knn_mlma_hate_speech_target_df['target_groups'])

# Check the distribution of the dependent variable in train vs test
mlma_hate_speech_target_train_dist = knn_mlma_hate_speech_target_df['tweet'].value_counts(normalize=True)
mlma_hate_speech_target_test_dist = knn_mlma_hate_speech_target_df['tweet'].value_counts(normalize=True)
print("Train Distribution:")
print(mlma_hate_speech_target_train_dist)
print("\nTest Distribution:")
print(mlma_hate_speech_target_test_dist)

Train Distribution:
tweet
@user retard                                                                                0.000531
@user nigger                                                                                0.000531
@user twat                                                                                  0.000443
@user faggot                                                                                0.000354
@user @user retard                                                                          0.000354
                                                                                              ...   
@user @user @user stop attacking journalists science - shithole- countries p @url           0.000089
@user @user u retarded daddy? butter enough. u ever potatoes butter retard                  0.000089
imagine yoongi working tongue technology hands tightly gripping hips fucks cunt wit @url    0.000089
retarded fellow beyond tolerance . @url                          

In [ ]:
pd.Series(mlma_hate_speech_target_y_train_titles).value_counts()

,count
target_groups,
1,6115
2,1765
0,1024
3,131


## Embed Text

The sentence transformer (in this case, SentenceBERT) allows us to see the embeddings, which are vector representations of the text.

In [ ]:
# Install the sentence transformers
# !pip install -U sentence-transformers

Here's a function that will convert our text into embeddings to lower dimensionality.

In [11]:
# Develop a get_embeddings function to get the embeddings using the SentenceBERT model to embed the text
    # The embeddings are the vector representations of the text
def get_embeddings(text):
    """
    Get the embeddings for the text using the SentenceBERT model.
    """
    # Load the SentenceBERT model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Get the embeddings
    embeddings = model.encode(text)

    return embeddings

In [12]:
pickle_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/pickle/'
results_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/results/'

Generate the embeddings for the title text as the dependent variables.

In [13]:
# Build merged text for each split
convabuse_X_train_titles = convabuse_X_train_titles.copy()
convabuse_X_test_titles  = convabuse_X_test_titles.copy()

convabuse_X_train_titles["merged"] = convabuse_X_train_titles.astype(str).agg(" ".join, axis=1)
convabuse_X_test_titles["merged"]  = convabuse_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
convabuse_X_train = get_embeddings(convabuse_X_train_titles["merged"].tolist())
convabuse_X_test  = get_embeddings(convabuse_X_test_titles["merged"].tolist())

# Align labels to the same indices
convabuse_y_train = convabuse_y_train_titles.reindex(convabuse_X_train_titles.index)
convabuse_y_test  = convabuse_y_test_titles.reindex(convabuse_X_test_titles.index)

# Convert to numpy for sklearn
convabuse_y_train = convabuse_y_train.to_numpy()
convabuse_y_test  = convabuse_y_test.to_numpy()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
# Pickle the embeddings
with open(f'{pickle_path}convabuse_X_train_combined_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_X_train, f)
with open(f'{pickle_path}convabuse_X_test_combined_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_X_test, f)
with open(f'{pickle_path}convabuse_y_train_combined_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_y_train, f)
with open(f'{pickle_path}convabuse_y_test_combined_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_y_test, f)

# Open the pickled embeddings
with open(f'{pickle_path}convabuse_X_train_combined_imbalanced.pkl', 'rb') as f:
    convabuse_X_train = pickle.load(f)
with open(f'{pickle_path}convabuse_X_test_combined_imbalanced.pkl', 'rb') as f:
    convabuse_X_test = pickle.load(f)
with open(f'{pickle_path}convabuse_y_train_combined_imbalanced.pkl', 'rb') as f:
    convabuse_y_train = pickle.load(f)
with open(f'{pickle_path}convabuse_y_test_combined_imbalanced.pkl', 'rb') as f:
    convabuse_y_test = pickle.load(f)

In [15]:
# Get the shapes
print(convabuse_X_train.shape)
print(convabuse_X_test.shape)
print(convabuse_y_train.shape)
print(convabuse_y_test.shape)
print(convabuse_X_train.shape[0] + convabuse_X_test.shape[0], "x", convabuse_X_train.shape[1])

(10061, 384)
(2516, 384)
(10061,)
(2516,)
12577 x 384


In [ ]:
print(type(convabuse_prev_agent_X_train_titles))

<class 'pandas.core.series.Series'>


In [ ]:
# Output the titles as a csv file
convabuse_X_train_titles.to_csv(f'{results_path}convabuse_X_train_titles_imbalanced.csv', index=False)
convabuse_X_test_titles.to_csv(f'{results_path}convabuse_X_test_titles_imbalanced.csv', index=False)
convabuse_y_train_titles.to_csv(f'{results_path}convabuse_y_train_titles_imbalanced.csv', index=False)
convabuse_y_test_titles.to_csv(f'{results_path}convabuse_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Build merged text for each split
convabuse_prev_agent_X_train_titles = convabuse_prev_agent_X_train_titles.copy().to_frame()
convabuse_prev_agent_X_test_titles  = convabuse_prev_agent_X_test_titles.copy().to_frame()

convabuse_prev_agent_X_train_titles["merged"] = convabuse_prev_agent_X_train_titles.astype(str).agg(" ".join, axis=1)
convabuse_prev_agent_X_test_titles["merged"]  = convabuse_prev_agent_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
convabuse_prev_agent_X_train = get_embeddings(convabuse_prev_agent_X_train_titles["merged"].tolist())
convabuse_prev_agent_X_test  = get_embeddings(convabuse_prev_agent_X_test_titles["merged"].tolist())

# Align labels to the same indices
convabuse_prev_agent_y_train = convabuse_prev_agent_y_train_titles.reindex(convabuse_prev_agent_X_train_titles.index)
convabuse_prev_agent_y_test  = convabuse_prev_agent_y_test_titles.reindex(convabuse_prev_agent_X_test_titles.index)

# Convert to numpy for sklearn
convabuse_prev_agent_y_train = convabuse_prev_agent_y_train.to_numpy()
convabuse_prev_agent_y_test  = convabuse_prev_agent_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}convabuse_prev_agent_X_train_imbalanced.pkl', 'wb') as a:
    pickle.dump(convabuse_prev_agent_X_train, a)
with open(f'{pickle_path}convabuse_prev_agent_X_test_imbalanced.pkl', 'wb') as b:
    pickle.dump(convabuse_prev_agent_X_test, b)
with open(f'{pickle_path}convabuse_prev_agent_y_train_imbalanced.pkl', 'wb') as c:
    pickle.dump(convabuse_prev_agent_y_train, c)
with open(f'{pickle_path}convabuse_prev_agent_y_test_imbalanced.pkl', 'wb') as d:
    pickle.dump(convabuse_prev_agent_y_test, d)

# Open the pickled embeddings
with open(f'{pickle_path}convabuse_prev_agent_X_train_imbalanced.pkl', 'rb') as e:
    convabuse_prev_agent_X_train = pickle.load(e)
with open(f'{pickle_path}convabuse_prev_agent_X_test_imbalanced.pkl', 'rb') as f:
    convabuse_prev_agent_X_test = pickle.load(f)
with open(f'{pickle_path}convabuse_prev_agent_y_train_imbalanced.pkl', 'rb') as g:
    convabuse_prev_agent_y_train = pickle.load(g)
with open(f'{pickle_path}convabuse_prev_agent_y_test_imbalanced.pkl', 'rb') as h:
    convabuse_prev_agent_y_test = pickle.load(h)

In [ ]:
# Output the titles as a csv file
convabuse_X_train_titles.to_csv(f'{results_path}convabuse_prev_agent_X_train_titles_imbalanced.csv', index=False)
convabuse_X_test_titles.to_csv(f'{results_path}convabuse_prev_agent_X_test_titles_imbalanced.csv', index=False)
convabuse_y_train_titles.to_csv(f'{results_path}convabuse_prev_agent_y_train_titles_imbalanced.csv', index=False)
convabuse_y_test_titles.to_csv(f'{results_path}convabuse_prev_agent_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Get the shapes
print(convabuse_prev_agent_X_train.shape)
print(convabuse_prev_agent_X_test.shape)
print(convabuse_prev_agent_y_train.shape)
print(convabuse_prev_agent_y_test.shape)
print(convabuse_prev_agent_X_train.shape[0] + convabuse_prev_agent_X_test.shape[0], "x", convabuse_prev_agent_X_train.shape[1])

(10061, 384)
(2516, 384)
(10061,)
(2516,)
12577 x 384


In [ ]:
# Build merged text for each split
convabuse_prev_user_X_train_titles = convabuse_prev_user_X_train_titles.copy().to_frame()
convabuse_prev_user_X_test_titles  = convabuse_prev_user_X_test_titles.copy().to_frame()

convabuse_prev_user_X_train_titles["merged"] = convabuse_prev_user_X_train_titles.astype(str).agg(" ".join, axis=1)
convabuse_prev_user_X_test_titles["merged"]  = convabuse_prev_user_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
convabuse_prev_user_X_train = get_embeddings(convabuse_prev_user_X_train_titles["merged"].tolist())
convabuse_prev_user_X_test  = get_embeddings(convabuse_prev_user_X_test_titles["merged"].tolist())

# Align labels to the same indices
convabuse_prev_user_y_train = convabuse_prev_agent_y_train_titles.reindex(convabuse_prev_user_X_train_titles.index)
convabuse_prev_user_y_test  = convabuse_prev_agent_y_test_titles.reindex(convabuse_prev_user_X_test_titles.index)

# Convert to numpy for sklearn
convabuse_prev_user_y_train = convabuse_prev_user_y_train.to_numpy()
convabuse_prev_user_y_test  = convabuse_prev_user_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}convabuse_prev_user_X_train_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_prev_user_X_train, f)
with open(f'{pickle_path}convabuse_prev_user_X_test_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_prev_user_X_test, f)
with open(f'{pickle_path}convabuse_prev_user_y_train_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_prev_user_y_train, f)
with open(f'{pickle_path}convabuse_prev_user_y_test_imbalanced.pkl', 'wb') as f:
    pickle.dump(convabuse_prev_user_y_test, f)

# Open the pickled embeddings
with open(f'{pickle_path}convabuse_prev_user_X_train_imbalanced.pkl', 'rb') as f:
    convabuse_prev_user_X_train = pickle.load(f)
with open(f'{pickle_path}convabuse_prev_user_X_test_imbalanced.pkl', 'rb') as f:
    convabuse_prev_user_X_test = pickle.load(f)
with open(f'{pickle_path}convabuse_prev_user_y_train_imbalanced.pkl', 'rb') as f:
    convabuse_prev_user_y_train = pickle.load(f)
with open(f'{pickle_path}convabuse_prev_user_y_test_imbalanced.pkl', 'rb') as f:
    convabuse_prev_user_y_test = pickle.load(f)

In [ ]:
# Get each length
print(len(convabuse_prev_user_X_train))
print(len(convabuse_prev_user_X_test))
print(len(convabuse_prev_user_y_train))
print(len(convabuse_prev_user_y_test))
print(len(convabuse_prev_user_X_train) + len(convabuse_prev_user_X_test), "x", convabuse_prev_user_X_train.shape[1])

10061
2516
10061
2516
12577 x 384


In [ ]:
# Output the titles as a csv file
convabuse_prev_user_X_train_titles.to_csv(f'{results_path}convabuse_prev_user_X_train_titles_imbalanced.csv', index=False)
convabuse_prev_user_X_test_titles.to_csv(f'{results_path}convabuse_prev_user_X_test_titles_imbalanced.csv', index=False)
convabuse_prev_user_y_train_titles.to_csv(f'{results_path}convabuse_prev_user_y_train_titles_imbalanced.csv', index=False)
convabuse_prev_user_y_test_titles.to_csv(f'{results_path}convabuse_prev_user_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Build merged text for each split
convabuse_agent_X_train_titles = convabuse_agent_X_train_titles.copy().to_frame()
convabuse_agent_X_test_titles  = convabuse_agent_X_test_titles.copy().to_frame()

convabuse_agent_X_train_titles["merged"] = convabuse_agent_X_train_titles.astype(str).agg(" ".join, axis=1)
convabuse_agent_X_test_titles["merged"]  = convabuse_agent_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
convabuse_agent_X_train = get_embeddings(convabuse_agent_X_train_titles["merged"].tolist())
convabuse_agent_X_test  = get_embeddings(convabuse_agent_X_test_titles["merged"].tolist())

# Align labels to the same indices
convabuse_agent_y_train = convabuse_agent_y_train_titles.reindex(convabuse_agent_X_train_titles.index)
convabuse_agent_y_test  = convabuse_agent_y_test_titles.reindex(convabuse_agent_X_test_titles.index)

# Convert to numpy for sklearn
convabuse_agent_y_train = convabuse_agent_y_train.to_numpy()
convabuse_agent_y_test  = convabuse_agent_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}convabuse_agent_X_train_imbalanced.pkl', 'wb') as a:
    pickle.dump(convabuse_agent_X_train, a)
with open(f'{pickle_path}convabuse_agent_X_test_imbalanced.pkl', 'wb') as b:
    pickle.dump(convabuse_agent_X_test, b)
with open(f'{pickle_path}convabuse_agent_y_train_imbalanced.pkl', 'wb') as c:
    pickle.dump(convabuse_agent_y_train, c)
with open(f'{pickle_path}convabuse_agent_y_test_imbalanced.pkl', 'wb') as d:
    pickle.dump(convabuse_agent_y_test, d)

# Open the pickled embeddings
with open(f'{pickle_path}convabuse_agent_X_train_imbalanced.pkl', 'rb') as e:
    convabuse_agent_X_train = pickle.load(e)
with open(f'{pickle_path}convabuse_agent_X_test_imbalanced.pkl', 'rb') as f:
    convabuse_agent_X_test = pickle.load(f)
with open(f'{pickle_path}convabuse_agent_y_train_imbalanced.pkl', 'rb') as g:
    convabuse_agent_y_train = pickle.load(g)
with open(f'{pickle_path}convabuse_agent_y_test_imbalanced.pkl', 'rb') as h:
    convabuse_agent_y_test = pickle.load(h)

In [ ]:
# Output the titles as a csv file
convabuse_agent_X_train_titles.to_csv(f'{results_path}convabuse_agent_X_train_titles_imbalanced.csv', index=False)
convabuse_agent_X_test_titles.to_csv(f'{results_path}convabuse_agent_X_test_titles_imbalanced.csv', index=False)
convabuse_agent_y_train_titles.to_csv(f'{results_path}convabuse_agent_y_train_titles_imbalanced.csv', index=False)
convabuse_agent_y_test_titles.to_csv(f'{results_path}convabuse_agent_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Get the shapes
print(convabuse_agent_X_train.shape)
print(convabuse_agent_X_test.shape)
print(convabuse_agent_y_train.shape)
print(convabuse_agent_y_test.shape)
print(convabuse_agent_X_train.shape[0] + convabuse_agent_X_test.shape[0], "x", convabuse_agent_X_train.shape[1])

(10061, 384)
(2516, 384)
(10061,)
(2516,)
12577 x 384


In [ ]:
# Build merged text for each split
convabuse_user_X_train_titles = convabuse_user_X_train_titles.copy().to_frame()
convabuse_user_X_test_titles  = convabuse_user_X_test_titles.copy().to_frame()

convabuse_user_X_train_titles["merged"] = convabuse_user_X_train_titles.astype(str).agg(" ".join, axis=1)
convabuse_user_X_test_titles["merged"]  = convabuse_user_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
convabuse_user_X_train = get_embeddings(convabuse_user_X_train_titles["merged"].tolist())
convabuse_user_X_test  = get_embeddings(convabuse_user_X_test_titles["merged"].tolist())

# Align labels to the same indices
convabuse_user_y_train = convabuse_user_y_train_titles.reindex(convabuse_user_X_train_titles.index)
convabuse_user_y_test  = convabuse_user_y_test_titles.reindex(convabuse_user_X_test_titles.index)

# Convert to numpy for sklearn
convabuse_user_y_train = convabuse_user_y_train.to_numpy()
convabuse_user_y_test  = convabuse_user_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}convabuse_user_X_train_imbalanced.pkl', 'wb') as a:
    pickle.dump(convabuse_user_X_train, a)
with open(f'{pickle_path}convabuse_user_X_test_imbalanced.pkl', 'wb') as b:
    pickle.dump(convabuse_user_X_test, b)
with open(f'{pickle_path}convabuse_user_y_train_imbalanced.pkl', 'wb') as c:
    pickle.dump(convabuse_user_y_train, c)
with open(f'{pickle_path}convabuse_user_y_test_imbalanced.pkl', 'wb') as d:
    pickle.dump(convabuse_user_y_test, d)

# Open the pickled embeddings
with open(f'{pickle_path}convabuse_user_X_train_imbalanced.pkl', 'rb') as e:
    convabuse_user_X_train = pickle.load(e)
with open(f'{pickle_path}convabuse_user_X_test_imbalanced.pkl', 'rb') as f:
    convabuse_user_X_test = pickle.load(f)
with open(f'{pickle_path}convabuse_user_y_train_imbalanced.pkl', 'rb') as g:
    convabuse_user_y_train = pickle.load(g)
with open(f'{pickle_path}convabuse_user_y_test_imbalanced.pkl', 'rb') as h:
    convabuse_user_y_test = pickle.load(h)

In [ ]:
# Output the titles as a csv file
convabuse_user_X_train_titles.to_csv(f'{results_path}convabuse_user_X_train_titles_imbalanced.csv', index=False)
convabuse_user_X_test_titles.to_csv(f'{results_path}convabuse_user_X_test_titles_imbalanced.csv', index=False)
convabuse_user_y_train_titles.to_csv(f'{results_path}convabuse_user_y_train_titles_imbalanced.csv', index=False)
convabuse_user_y_test_titles.to_csv(f'{results_path}convabuse_user_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Build merged text for each split
dghs_target_X_train_titles = dghs_target_X_train_titles.copy().to_frame()
dghs_target_X_test_titles  = dghs_target_X_test_titles.copy().to_frame()

dghs_target_X_train_titles["merged"] = dghs_target_X_train_titles.astype(str).agg(" ".join, axis=1)
dghs_target_X_test_titles["merged"]  = dghs_target_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
dghs_target_X_train = get_embeddings(dghs_target_X_train_titles["merged"].tolist())
dghs_target_X_test  = get_embeddings(dghs_target_X_test_titles["merged"].tolist())

# Align labels to the same indices
dghs_target_y_train = dghs_target_y_train_titles.reindex(dghs_target_X_train_titles.index)
dghs_target_y_test  = dghs_target_y_test_titles.reindex(dghs_target_X_test_titles.index)

# Convert to numpy for sklearn
dghs_target_y_train = dghs_target_y_train.to_numpy()
dghs_target_y_test  = dghs_target_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}dghs_target_X_train_imbalanced.pkl', 'wb') as a:
    pickle.dump(dghs_target_X_train, a)
with open(f'{pickle_path}dghs_target_X_test_imbalanced.pkl', 'wb') as b:
    pickle.dump(dghs_target_X_test, b)
with open(f'{pickle_path}dghs_target_y_train_imbalanced.pkl', 'wb') as c:
    pickle.dump(dghs_target_y_train, c)
with open(f'{pickle_path}dghs_target_y_test_imbalanced.pkl', 'wb') as d:
    pickle.dump(dghs_target_y_test, d)

# Open the pickled embeddings
with open(f'{pickle_path}dghs_target_X_train_imbalanced.pkl', 'rb') as e:
    dghs_target_X_train = pickle.load(e)
with open(f'{pickle_path}dghs_target_X_test_imbalanced.pkl', 'rb') as f:
    dghs_target_X_test = pickle.load(f)
with open(f'{pickle_path}dghs_target_y_train_imbalanced.pkl', 'rb') as g:
    dghs_target_y_train = pickle.load(g)
with open(f'{pickle_path}dghs_target_y_test_imbalanced.pkl', 'rb') as h:
    dghs_target_y_test = pickle.load(h)

In [ ]:
# Output the titles as a csv file
dghs_target_X_train_titles.to_csv(f'{results_path}dghs_target_X_train_titles_imbalanced.csv', index=False)
dghs_target_X_test_titles.to_csv(f'{results_path}dghs_target_X_test_titles_imbalanced.csv', index=False)
dghs_target_y_train_titles.to_csv(f'{results_path}dghs_target_y_train_titles_imbalanced.csv', index=False)
dghs_target_y_test_titles.to_csv(f'{results_path}dghs_target_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Get the shapes
print(dghs_target_X_train_titles.shape)
print(dghs_target_X_test_titles.shape)
print(dghs_target_y_train_titles.shape)
print(dghs_target_y_test_titles.shape)
print(dghs_target_X_train_titles.shape[0] + dghs_target_X_test_titles.shape[0])

(32749, 2)
(8188, 2)
(32749,)
(8188,)
40937


In [ ]:
# Build merged text for each split
dghs_label_X_train_titles = dghs_label_X_train_titles.copy().to_frame()
dghs_label_X_test_titles  = dghs_label_X_test_titles.copy().to_frame()

dghs_label_X_train_titles["merged"] = dghs_label_X_train_titles.astype(str).agg(" ".join, axis=1)
dghs_label_X_test_titles["merged"]  = dghs_label_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
dghs_label_X_train = get_embeddings(dghs_label_X_train_titles["merged"].tolist())
dghs_label_X_test  = get_embeddings(dghs_label_X_test_titles["merged"].tolist())

# Align labels to the same indices
dghs_label_y_train = dghs_label_y_train_titles.reindex(dghs_label_X_train_titles.index)
dghs_label_y_test  = dghs_label_y_test_titles.reindex(dghs_label_X_test_titles.index)

# Convert to numpy for sklearn
dghs_label_y_train = dghs_label_y_train.to_numpy()
dghs_label_y_test  = dghs_label_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}dghs_label_X_train_imbalanced.pkl', 'wb') as a:
    pickle.dump(dghs_label_X_train, a)
with open(f'{pickle_path}dghs_label_X_test_imbalanced.pkl', 'wb') as b:
    pickle.dump(dghs_label_X_test, b)
with open(f'{pickle_path}dghs_label_y_train_imbalanced.pkl', 'wb') as c:
    pickle.dump(dghs_label_y_train, c)
with open(f'{pickle_path}dghs_label_y_test_imbalanced.pkl', 'wb') as d:
    pickle.dump(dghs_label_y_test, d)

# Open the pickled embeddings
with open(f'{pickle_path}dghs_label_X_train_imbalanced.pkl', 'rb') as e:
    dghs_label_X_train = pickle.load(e)
with open(f'{pickle_path}dghs_label_X_test_imbalanced.pkl', 'rb') as f:
    dghs_label_X_test = pickle.load(f)
with open(f'{pickle_path}dghs_label_y_train_imbalanced.pkl', 'rb') as g:
    dghs_label_y_train = pickle.load(g)
with open(f'{pickle_path}dghs_label_y_test_imbalanced.pkl', 'rb') as h:
    dghs_label_y_test = pickle.load(h)

In [ ]:
# Output the titles as a csv file
dghs_label_X_train_titles.to_csv(f'{results_path}dghs_label_X_train_titles_imbalanced.csv', index=False)
dghs_label_X_test_titles.to_csv(f'{results_path}dghs_label_X_test_titles_imbalanced.csv', index=False)
dghs_label_y_train_titles.to_csv(f'{results_path}dghs_label_y_train_titles_imbalanced.csv', index=False)
dghs_label_y_test_titles.to_csv(f'{results_path}dghs_label_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Get the shapes
print(dghs_label_X_train_titles.shape)
print(dghs_label_X_test_titles.shape)
print(dghs_label_y_train_titles.shape)
print(dghs_label_y_test_titles.shape)
print(dghs_label_X_train_titles.shape[0] + dghs_label_X_test_titles.shape[0])

(32749, 2)
(8188, 2)
(32749,)
(8188,)
40937


In [ ]:
# Build merged text for each split
mlma_hate_speech_target_X_train_titles = mlma_hate_speech_target_X_train_titles.copy().to_frame()
mlma_hate_speech_target_X_test_titles  = mlma_hate_speech_target_X_test_titles.copy().to_frame()

mlma_hate_speech_target_X_train_titles["merged"] = mlma_hate_speech_target_X_train_titles.astype(str).agg(" ".join, axis=1)
mlma_hate_speech_target_X_test_titles["merged"]  = mlma_hate_speech_target_X_test_titles.astype(str).agg(" ".join, axis=1)

# Embed
mlma_hate_speech_target_X_train = get_embeddings(mlma_hate_speech_target_X_train_titles["merged"].tolist())
mlma_hate_speech_target_X_test  = get_embeddings(mlma_hate_speech_target_X_test_titles["merged"].tolist())

# Align labels to the same indices
mlma_hate_speech_target_y_train = mlma_hate_speech_target_y_train_titles.reindex(mlma_hate_speech_target_X_train_titles.index)
mlma_hate_speech_target_y_test  = mlma_hate_speech_target_y_test_titles.reindex(mlma_hate_speech_target_X_test_titles.index)

# Convert to numpy for sklearn
mlma_hate_speech_target_y_train = mlma_hate_speech_target_y_train.to_numpy()
mlma_hate_speech_target_y_test  = mlma_hate_speech_target_y_test.to_numpy()

In [ ]:
# Pickle the embeddings
with open(f'{pickle_path}mlma_hate_speech_target_X_train_imbalanced.pkl', 'wb') as a:
    pickle.dump(mlma_hate_speech_target_X_train, a)
with open(f'{pickle_path}mlma_hate_speech_target_X_test_imbalanced.pkl', 'wb') as b:
    pickle.dump(mlma_hate_speech_target_X_test, b)
with open(f'{pickle_path}mlma_hate_speech_target_y_train_imbalanced.pkl', 'wb') as c:
    pickle.dump(mlma_hate_speech_target_y_train, c)
with open(f'{pickle_path}mlma_hate_speech_target_y_test_imbalanced.pkl', 'wb') as d:
    pickle.dump(mlma_hate_speech_target_y_test, d)

# Open the pickled embeddings
with open(f'{pickle_path}mlma_hate_speech_target_X_train_imbalanced.pkl', 'rb') as e:
    mlma_hate_speech_target_X_train = pickle.load(e)
with open(f'{pickle_path}mlma_hate_speech_target_X_test_imbalanced.pkl', 'rb') as f:
    mlma_hate_speech_target_X_test = pickle.load(f)
with open(f'{pickle_path}mlma_hate_speech_target_y_train_imbalanced.pkl', 'rb') as g:
    mlma_hate_speech_target_y_train = pickle.load(g)
with open(f'{pickle_path}mlma_hate_speech_target_y_test_imbalanced.pkl', 'rb') as h:
    mlma_hate_speech_target_y_test = pickle.load(h)

In [ ]:
# Output the titles as a csv file
mlma_hate_speech_target_X_train_titles.to_csv(f'{results_path}mlma_hate_speech_target_X_train_titles_imbalanced.csv', index=False)
mlma_hate_speech_target_X_test_titles.to_csv(f'{results_path}mlma_hate_speech_target_X_test_titles_imbalanced.csv', index=False)
mlma_hate_speech_target_y_train_titles.to_csv(f'{results_path}mlma_hate_speech_target_y_train_titles_imbalanced.csv', index=False)
mlma_hate_speech_target_y_test_titles.to_csv(f'{results_path}mlma_hate_speech_target_y_test_titles_imbalanced.csv', index=False)

In [ ]:
# Get the shapes
print(mlma_hate_speech_target_X_train_titles.shape)
print(mlma_hate_speech_target_X_test_titles.shape)
print(mlma_hate_speech_target_y_train_titles.shape)
print(mlma_hate_speech_target_y_test_titles.shape)
print(mlma_hate_speech_target_X_train_titles.shape[0] + mlma_hate_speech_target_X_test_titles.shape[0])

(9035, 2)
(2259, 2)
(9035,)
(2259,)
11294
